In [ ]:
# ============================
# 1. Imports
# ============================

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score


print("Libraries imported successfully")

Libraries imported successfully


In [2]:
# ============================
# 2. Load Data
# ============================

KAGGLE_DATASET_URL = "/kaggle/input/competitions/playground-series-s6e9/"

def find_data_dir() -> Path:
    candidates = []
    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd,
        cwd / "notebook",
        cwd.parent,
        cwd.parent / "notebook",
    ])

    for base in candidates:
        for candidate in [base / "input", base.parent / "input"]:
            if candidate.exists() and (candidate / "train.csv").exists() and (candidate / "test.csv").exists():
                return candidate

    for kaggle_path in [
        Path(KAGGLE_DATASET_URL),
        Path("/kaggle/input/train.csv").parent,
    ]:
        if kaggle_path.exists() and (kaggle_path / "train.csv").exists() and (kaggle_path / "test.csv").exists():
            return kaggle_path

    return Path(KAGGLE_DATASET_URL)


data_dir = find_data_dir()

train = pd.read_csv(data_dir / "train.csv")
test = pd.read_csv(data_dir / "test.csv")
sample_submission = pd.read_csv(data_dir / "sample_submission.csv")

print(f"Train: {train.shape}")
print(f"Test : {test.shape}")

Train: (668665, 15)
Test : (286571, 14)


In [3]:
# ============================
# 3. Identify ID and Target
# ============================

id_column = sample_submission.columns[0]
target_column = sample_submission.columns[1]

print(f"ID column     : {id_column}")
print(f"Target column : {target_column}")


ID column     : id
Target column : Will_Buy_EV


In [4]:
# ============================
# 4. Prepare Features
# ============================

X = train.drop(columns=target_column)
y = train[target_column]

X_test = test.copy()

# Remove ID automatically
X = X.drop(columns=id_column, errors="ignore")
X_test = X_test.drop(columns=id_column, errors="ignore")

In [5]:
# ============================
# 5. Detect Column Types
# ============================

numeric_columns = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_columns = X.select_dtypes(
    exclude=np.number
).columns.tolist()

print(f"Numerical columns   : {len(numeric_columns)}")
print(f"Categorical columns : {len(categorical_columns)}")

Numerical columns   : 7
Categorical columns : 6


In [6]:
# ============================
# 6. Preprocessing
# ============================

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    (
        "encoder",
        OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1
        )
    )
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_columns),
        ("categorical", categorical_pipeline, categorical_columns)
    ]
)

In [7]:
# ============================
# 7. Model
# ============================

model = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=500,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

In [8]:
# ============================
# 8. Pipeline
# ============================

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

In [9]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# =========================================================
# Method 1: cross_val_predict
# =========================================================

oof_1 = cross_val_predict(
    pipeline,
    X,
    y,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

auc_1 = roc_auc_score(y, oof_1)


# =========================================================
# Method 2: Manual loop
# =========================================================

from sklearn.base import clone
from tqdm.auto import tqdm

oof_2 = np.zeros(len(X))

for fold, (train_idx, valid_idx) in tqdm(
    enumerate(cv.split(X, y), start=1),
    total=cv.n_splits,
    desc="OOF",
    unit="fold"
):
    fold_pipeline = clone(pipeline)

    fold_pipeline.fit(
        X.iloc[train_idx],
        y.iloc[train_idx]
    )

    oof_2[valid_idx] = fold_pipeline.predict_proba(
        X.iloc[valid_idx]
    )[:, 1]


auc_2 = roc_auc_score(y, oof_2)


# =========================================================
# Compare
# =========================================================

print(f"\nAUC cross_val_predict : {auc_1:.15f}")
print(f"AUC manual OOF        : {auc_2:.15f}")

print(
    "\nMaximum prediction difference:",
    np.max(np.abs(oof_1 - oof_2))
)


OOF:   0%|          | 0/5 [00:00<?, ?fold/s]


AUC cross_val_predict : 0.941407813895485
AUC manual OOF        : 0.941407813895485

Maximum prediction difference: 0.0


In [16]:
# =========================================================
# Diagnostic: OOF vs training AUC
# =========================================================

pipeline.fit(X, y)

train_predictions = pipeline.predict_proba(X)[:, 1]

train_auc = roc_auc_score(y, train_predictions)
oof_auc = roc_auc_score(y, oof_1)

print("\n==============================")
print("AUC DIAGNOSTIC")
print("==============================")
print(f"OOF AUC   : {oof_auc:.15f}")
print(f"Train AUC : {train_auc:.15f}")

print("\nPrediction correlation:")
print(np.corrcoef(oof_1, train_predictions)[0, 1])


AUC DIAGNOSTIC
OOF AUC   : 0.941407813895485
Train AUC : 0.942466347785033

Prediction correlation:
0.9984374976705498


In [17]:
# ============================
# 12. Test Predictions
# ============================

test_predictions = pipeline.predict_proba(
    X_test
)[:, 1]

In [18]:
# ============================
# 13. Create Submission
# ============================

submission = sample_submission.copy()

submission[target_column] = test_predictions

output_path = "submission.parquet"

submission.to_parquet(
    output_path,
    index=False
)

In [19]:
# ============================
# 14. Results
# ============================

print("\nSubmission preview:")
print(submission.head())

print("\nPrediction statistics:")
print(submission[target_column].describe())


Submission preview:
       id  Will_Buy_EV
0  668665     0.008471
1  668666     0.020125
2  668667     0.007168
3  668668     0.004098
4  668669     0.016753

Prediction statistics:
count    286571.000000
mean          0.174705
std           0.267980
min           0.000103
25%           0.003171
50%           0.018370
75%           0.275118
max           0.955892
Name: Will_Buy_EV, dtype: float64
